In [0]:
catalog = "neobank"
from pyspark.sql import functions as F
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.branch_performance AS 
WITH cust AS (
    SELECT branch_code,
           COUNT(*) AS total_customers
    FROM {catalog}.silver.customers
    GROUP BY branch_code
),
acc AS (
    SELECT branch_code, COUNT(*) AS total_accounts, SUM(balance) AS total_deposits
    FROM {catalog}.silver.accounts 
    GROUP BY branch_code
),
txn AS (
    SELECT a.branch_code, COUNT(*) AS total_transactions, SUM(t.amount) AS total_transaction_amount
    FROM {catalog}.silver.transactions t
    JOIN {catalog}.silver.accounts a ON t.account_id = a.account_id
    GROUP BY a.branch_code
)
SELECT b.branch_code, b.branch_name, b.city, b.state, b.region,
       COALESCE(c.total_customers, 0)          AS total_customers,
       COALESCE(a.total_accounts, 0)           AS total_accounts,
       COALESCE(a.total_deposits, 0)           AS total_deposits,
       COALESCE(t.total_transactions, 0)       AS total_transactions,
       COALESCE(t.total_transaction_amount, 0) AS total_transaction_amount
FROM {catalog}.silver.branches b
LEFT JOIN cust c ON b.branch_code = c.branch_code
LEFT JOIN acc  a ON b.branch_code = a.branch_code
LEFT JOIN txn  t ON b.branch_code = t.branch_code
""")

In [0]:
high_cut = 469481.11
med_cut  = 269635.01

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.customer_360 AS
WITH acc AS (
  SELECT customer_id, COUNT(*) AS total_accounts, SUM(balance) AS total_balance
  FROM {catalog}.silver.accounts GROUP BY customer_id),
txn AS (
  SELECT a.customer_id, COUNT(*) AS total_transactions, SUM(t.amount) AS total_transaction_amount
  FROM {catalog}.silver.transactions t
  JOIN {catalog}.silver.accounts a ON t.account_id = a.account_id
  GROUP BY a.customer_id),
bureau AS (
  SELECT customer_id, credit_score, risk_grade, external_active_loans, external_overdue_amount
  FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY bureau_pull_date DESC) AS rn
        FROM {catalog}.silver.credit_bureau_reports)
  WHERE rn = 1)
SELECT c.customer_id,
       CONCAT(c.first_name, ' ', c.last_name) AS customer_name,
       c.kyc_status, c.branch_code, b.branch_name,
       COALESCE(a.total_accounts, 0)            AS total_accounts,
       COALESCE(a.total_balance, 0)             AS total_balance,
       COALESCE(t.total_transactions, 0)        AS total_transactions,
       COALESCE(t.total_transaction_amount, 0)  AS total_transaction_amount,
       br.credit_score, br.risk_grade, br.external_active_loans, br.external_overdue_amount,
       CASE WHEN COALESCE(a.total_accounts, 0) = 0 THEN 'No Account'
            WHEN COALESCE(a.total_balance, 0) >= {high_cut} THEN 'High Value'
            WHEN COALESCE(a.total_balance, 0) >= {med_cut}  THEN 'Medium Value'
            ELSE 'Low Value' END AS customer_segment
FROM {catalog}.silver.customers c
LEFT JOIN {catalog}.silver.branches b ON c.branch_code = b.branch_code
LEFT JOIN acc a     ON c.customer_id = a.customer_id
LEFT JOIN txn t     ON c.customer_id = t.customer_id
LEFT JOIN bureau br ON c.customer_id = br.customer_id
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.risk_customer_summary AS
SELECT risk_grade,
       COUNT(DISTINCT customer_id)  AS total_customers,
       ROUND(AVG(credit_score), 1)  AS avg_credit_score,
       SUM(external_active_loans)   AS total_active_loans,
       SUM(external_overdue_amount) AS total_overdue_amount
FROM {catalog}.gold.customer_360
WHERE risk_grade IS NOT NULL
GROUP BY risk_grade
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.daily_bank_kpi AS
WITH daily AS (
  SELECT DATE(txn_timestamp) AS txn_date,
         COUNT(*) AS total_transactions, SUM(amount) AS total_transaction_amount
  FROM {catalog}.silver.transactions GROUP BY DATE(txn_timestamp)),
bank AS (
  SELECT (SELECT COUNT(*) FROM {catalog}.silver.customers) AS total_customers,
         (SELECT COUNT(*) FROM {catalog}.silver.accounts)  AS total_accounts,
         (SELECT SUM(balance) FROM {catalog}.silver.accounts) AS total_balance,
         (SELECT ROUND(AVG(credit_score), 1) FROM {catalog}.silver.credit_bureau_reports) AS avg_credit_score,
         (SELECT COUNT(DISTINCT customer_id) FROM {catalog}.silver.credit_bureau_reports
          WHERE risk_grade = 'HIGH') AS high_risk_customers)
SELECT d.*, b.* FROM daily d CROSS JOIN bank b
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.transaction_channel_summary AS
SELECT DATE(t.txn_timestamp) AS txn_date,
       g.gateway_name, g.device_type,
       COUNT(*) AS total_transactions,
       SUM(CASE WHEN g.gateway_status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_transactions,
       SUM(CASE WHEN g.gateway_status = 'FAILED'  THEN 1 ELSE 0 END) AS failed_transactions,
       SUM(CASE WHEN g.gateway_status = 'PENDING' THEN 1 ELSE 0 END) AS pending_transactions,
       ROUND(AVG(g.processing_time_ms), 1) AS avg_processing_time_ms
FROM {catalog}.silver.transactions t
JOIN {catalog}.silver.payment_gateway_logs g ON t.txn_id = g.txn_id
GROUP BY DATE(t.txn_timestamp), g.gateway_name, g.device_type
""")

In [0]:
tx  = spark.table(f"{catalog}.silver.transactions").count()
agg = spark.sql(f"SELECT SUM(total_transactions) FROM {catalog}.gold.branch_performance").first()[0]
assert tx == agg, f"Branch totals do not reconcile: silver={tx}, gold={agg}"

chan = spark.sql(f"SELECT SUM(total_transactions) FROM {catalog}.gold.transaction_channel_summary").first()[0]
assert tx == chan, f"Channel totals do not reconcile: silver={tx}, gold={chan}"

assert spark.table(f"{catalog}.gold.customer_360").count() == spark.table(f"{catalog}.silver.customers").count(), "customer_360 row count mismatch"
print("Gold reconciliation passed")